In [3]:
from google.colab import drive
drive.mount('/content/drive')

utils_code = '''
import subprocess
subprocess.run(["pip", "install", "fiftyone", "-q"], check=True)

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import pickle
import fiftyone as fo
import os

def load_dataset():
    with open("/content/drive/MyDrive/ML_aircraft_recognition/dataset.pkl", "rb") as f:
        samples_data = pickle.load(f)
    dataset = fo.Dataset(name="fgvc-aircraft", overwrite=True)
    dataset.add_samples([
        fo.Sample(filepath=s["filepath"], ground_truth=fo.Classification(label=s["label"]))
        for s in samples_data
    ])
    print(f"Loaded {len(dataset)} samples")
    return dataset

def save_dataset(dataset):
    samples_data = [
        {"filepath": s.filepath, "label": s.ground_truth.label}
        for s in dataset
    ]
    with open("/content/drive/MyDrive/ML_aircraft_recognition/dataset.pkl", "wb") as f:
        pickle.dump(samples_data, f)
    print(f"Saved {len(samples_data)} samples to pickle")

class AircraftDataset(Dataset):
    def __init__(self, fo_dataset, transform=None):
        self.samples   = list(fo_dataset)
        self.labels    = sorted(fo_dataset.distinct("ground_truth.label"))
        self.label2idx = {l: i for i, l in enumerate(self.labels)}
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img    = Image.open(sample.filepath).convert("RGB")
        label  = self.label2idx[sample.ground_truth.label]
        if self.transform:
            img = self.transform(img)
        return img, label

    def num_classes(self):
        return len(self.labels)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_dataloaders(dataset, batch_size=32):
    full_dataset = AircraftDataset(dataset, transform=train_transform)
    num_classes  = full_dataset.num_classes()

    train_size = int(0.70 * len(full_dataset))
    val_size   = int(0.15 * len(full_dataset))
    test_size  = len(full_dataset) - train_size - val_size

    train_set, val_set, test_set = random_split(
        full_dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )
    val_set.dataset.transform  = val_transform
    test_set.dataset.transform = val_transform

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=2)

    print(f"Train: {train_size} | Val: {val_size} | Test: {test_size} | Classes: {num_classes}")
    return train_loader, val_loader, test_loader, num_classes, train_size, val_size, test_size

def build_model(num_classes, device):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(device)

SAVE_PATH = "/content/drive/MyDrive/ML_aircraft_recognition/best_model.pth"

def save_model(model):
    torch.save(model.state_dict(), SAVE_PATH)
    print(f"Model saved to {SAVE_PATH}")

def load_model(num_classes, device):
    model = build_model(num_classes, device)
    model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
    model.eval()
    print(f"Model loaded from {SAVE_PATH}")
    return model

def build_resnet(num_classes, device):
    from torchvision.models import resnet50, ResNet50_Weights
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

RESNET_SAVE_PATH = "/content/drive/MyDrive/ML_aircraft_recognition/best_resnet.pth"

def save_resnet(model):
    torch.save(model.state_dict(), RESNET_SAVE_PATH)
    print(f"ResNet saved to {RESNET_SAVE_PATH}")

def load_resnet(num_classes, device):
    model = build_resnet(num_classes, device)
    model.load_state_dict(torch.load(RESNET_SAVE_PATH, map_location=device))
    model.eval()
    print(f"ResNet loaded from {RESNET_SAVE_PATH}")
    return model
'''

with open("/content/drive/MyDrive/ML_aircraft_recognition/utils.py", "w") as f:
    f.write(utils_code)

print("✅ utils.py saved to Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ utils.py saved to Drive!
